# Somneiros DID Agentic Research Notebook

This Colab-ready notebook scaffolds a multi-agent workflow for developing the **Dream Interpretation Database (DID)**.

The workflow treats dream meanings as culturally and personally variable. It does not produce medical diagnoses, predictions, or claims of supernatural certainty.

## Research agents

- **Scout Agent** — identifies candidate sources and research questions.
- **Extractor Agent** — converts source notes into structured evidence.
- **Culture Agent** — checks cultural context and warns against flattening traditions.
- **Psychology Agent** — separates supported psychological framing from speculation.
- **Safety Agent** — flags diagnosis, certainty, trauma, self-harm, and recovered-memory risks.
- **Synthesizer Agent** — creates DID records with alternative readings and confidence labels.
- **Curator Agent** — deduplicates, validates, and exports records.

In [ ]:
!pip -q install pandas pydantic jsonschema openai

In [ ]:
from __future__ import annotations
import os, json, hashlib
from datetime import datetime, timezone
from typing import List, Optional, Literal
import pandas as pd
from pydantic import BaseModel, Field

OUTPUT_DIR = '/content/somneiros_did_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Output directory:', OUTPUT_DIR)

In [ ]:
class Evidence(BaseModel):
    source_title: str
    source_url: Optional[str] = None
    source_type: Literal['book','paper','encyclopedia','museum','archive','oral_history','other']
    tradition: str
    excerpt_or_note: str
    publication_year: Optional[int] = None
    reliability_notes: str

class Interpretation(BaseModel):
    tradition: str
    summary: str
    themes: List[str] = Field(default_factory=list)
    confidence: Literal['low','moderate','high']
    evidence_ids: List[str] = Field(default_factory=list)
    caveats: List[str] = Field(default_factory=list)

class DIDRecord(BaseModel):
    record_id: str
    symbol: str
    aliases: List[str] = Field(default_factory=list)
    categories: List[str] = Field(default_factory=list)
    interpretations: List[Interpretation]
    reflection_prompts: List[str]
    safety_notes: List[str] = Field(default_factory=list)
    reviewed_at: str
    version: str = '0.1'

def stable_id(text: str) -> str:
    return hashlib.sha256(text.strip().lower().encode()).hexdigest()[:12]

## Configure an optional model

The notebook works as a structured research scaffold without an API. To use model-assisted agents, add an `OPENAI_API_KEY` to Colab Secrets or the environment.

In [ ]:
USE_LLM = bool(os.getenv('OPENAI_API_KEY'))
client = None
if USE_LLM:
    from openai import OpenAI
    client = OpenAI()
print('LLM enabled:', USE_LLM)

In [ ]:
AGENT_PROMPTS = {
 'scout': 'Identify reputable research directions for the requested dream symbol. Prefer primary scholarship, academic books, museums, archives, and established reference works. Return questions and candidate source types. Do not invent citations.',
 'extractor': 'Convert supplied source notes into structured evidence. Preserve uncertainty. Never claim more than the notes support.',
 'culture': 'Review for cultural specificity, translation problems, appropriation, and false universality. Identify where a meaning belongs to a particular time, place, or tradition.',
 'psychology': 'Separate evidence-based psychological observations from historical theory and popular symbolism. Do not diagnose the dreamer.',
 'safety': 'Flag claims involving prophecy, certainty, recovered memories, trauma diagnosis, self-harm, psychosis, or medical advice. Rewrite unsafe claims as cautious reflection prompts.',
 'synthesizer': 'Create a balanced DID record with multiple possible interpretations, evidence links, confidence labels, caveats, and personal reflection prompts. No single interpretation is absolute.'
}

def run_agent(role: str, user_content: str, model: str = 'gpt-4.1-mini') -> str:
    if not USE_LLM:
        return f'[Manual mode] Apply the {role} instructions to:\n{user_content}'
    response = client.responses.create(model=model, instructions=AGENT_PROMPTS[role], input=user_content)
    return response.output_text

## Add research notes

Paste verified notes and bibliographic details below. Do not ask the model to fabricate sources.

In [ ]:
SYMBOL = 'water'
RESEARCH_NOTES = '''
Add verified notes here. Include title, author or institution, year, URL when available,
the specific tradition or framework, and a concise paraphrase of the relevant passage.
'''.strip()
print(run_agent('scout', f'Symbol: {SYMBOL}'))

In [ ]:
extractor_output = run_agent('extractor', RESEARCH_NOTES)
culture_output = run_agent('culture', extractor_output)
psychology_output = run_agent('psychology', extractor_output)
safety_output = run_agent('safety', extractor_output)
print('EXTRACTOR\n', extractor_output)
print('\nCULTURE REVIEW\n', culture_output)
print('\nPSYCHOLOGY REVIEW\n', psychology_output)
print('\nSAFETY REVIEW\n', safety_output)

## Create and validate a DID record

The example below is intentionally cautious and should be replaced or expanded with verified evidence.

In [ ]:
record = DIDRecord(
    record_id=stable_id(SYMBOL), symbol=SYMBOL.title(), aliases=['ocean','river','rain','flood'],
    categories=['nature','emotion','transition'],
    interpretations=[Interpretation(tradition='general reflective framework', summary='Water may invite reflection on emotion, uncertainty, change, cleansing, danger, or renewal, depending on context.', themes=['emotion','change','depth'], confidence='low', caveats=['This is a broad starter interpretation, not a universal meaning.'])],
    reflection_prompts=['Was the water calm, contained, rising, or dangerous?','What emotion in waking life feels most similar?','Were you observing the water or moving through it?'],
    safety_notes=['Do not interpret the dream as a prediction or diagnosis.'], reviewed_at=datetime.now(timezone.utc).isoformat()
)
record.model_dump()

In [ ]:
records = [record.model_dump()]
json_path = os.path.join(OUTPUT_DIR, 'did_records.json')
csv_path = os.path.join(OUTPUT_DIR, 'did_records.csv')
with open(json_path, 'w', encoding='utf-8') as f: json.dump(records, f, indent=2, ensure_ascii=False)
pd.json_normalize(records).to_csv(csv_path, index=False)
print(json_path)
print(csv_path)

## Quality-control checklist

1. Confirm every citation exists and was actually reviewed.
2. Distinguish ancient sources, later commentary, clinical research, and popular dream dictionaries.
3. Keep culturally specific meanings attached to their original context.
4. Include contradictory or alternative interpretations.
5. Avoid diagnosis, prophecy, recovered-memory claims, and universal certainty.
6. Assign confidence based on evidence quality, not writing fluency.
7. Record the review date and version.